In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn

In [2]:
SMALL_CAP = ["600169.SH", "000049.SZ", "600428.SH", "002064.SZ", "000501.SZ", "603000.SH", "000712.SZ", "600580.SH", "600017.SH", "603355.SH"]
MID_CAP = ["600138.SH", "000718.SZ", "002444.SZ", "600597.SH", "002285.SZ", "600729.SH", "000062.SZ", "002463.SZ", "002603.SZ", "603883.SH"]
LARGE_CAP = ["600718.SH", "600507.SH", "002583.SZ", "601718.SH", "600835.SH", "002268.SZ", "002092.SZ", "600098.SH", "002340.SZ", "300001.SZ"]

ASSETS = SMALL_CAP + MID_CAP + LARGE_CAP

In [3]:
DATA = Path("./data")
NUM_ASSETS = len(ASSETS)

# Prices and returns
daily_prices = pd.read_pickle(DATA / "daily_adj_close_post_20260630.pkl")
monthly_returns = pd.read_pickle(DATA / "monthly_returns.pkl")
prices = pd.read_hdf(DATA / "price.h5", key="price")

# Trading characteristics
daily_amount = pd.read_pickle(DATA / "daily_amount.pkl")
daily_turnover = pd.read_pickle(DATA / "daily_turnover.pkl")
daily_volume = pd.read_pickle(DATA / "daily_volume.pkl")

# Company characteristics
mcap = pd.read_pickle(DATA / "mcap.pkl")
industry = pd.read_pickle(DATA / "ind_code.pkl")

# Factor returns
ff5 = pd.read_csv(DATA / "FF5.csv", index_col=0, parse_dates=True)
hxz = pd.read_csv(DATA / "HXZ.csv", index_col=0, parse_dates=True)

# Risk-free rates
risk_free = pd.read_csv(DATA / "risk_free.csv")
shibor = pd.read_pickle(DATA / "shibor_rates.pkl")

# Stock selection and trading constraints
blacklist = pd.read_pickle(DATA / "BLACKLIST.pkl")
untradable = pd.read_pickle(DATA / "UNTRADABLE.pkl")
csi500_mask = pd.read_pickle(DATA / "csi500_mask_monthly.pkl")

In [4]:
daily_prices.tail()

order_book_id,000001.SZ,000002.SZ,000003.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,...,688712.SH,688781.SH,688785.SH,688797.SH,688808.SH,688811.SH,688813.SH,688816.SH,688818.SH,688820.SH
date,,,,,,,,,,,,,,,,,,,,,
2026-06-24,1931.9482,431.5232,NaN,1.6485,NaN,546.2453,164.2401,64.4415,79.7935,20.1383,...,39.38,54.33,454.7999,585.00,2586.03,30.09,222.20,62.2025,64.03,213.98
2026-06-25,1915.4044,428.6173,NaN,1.5308,NaN,534.6230,157.0102,62.5697,79.0725,19.1894,...,37.81,52.65,436.9125,574.44,2700.00,28.74,243.80,59.4221,63.60,210.50
2026-06-26,1880.4786,437.3349,NaN,1.6485,NaN,520.8216,152.5517,60.1632,74.3858,18.7676,...,34.40,49.05,450.9218,545.07,2300.00,29.09,230.90,56.8726,70.55,190.25
2026-06-29,1882.3168,441.6938,NaN,1.8251,NaN,532.4439,148.0933,62.8371,73.7850,17.9241,...,37.00,49.69,475.9842,598.28,2350.00,28.33,211.99,57.4146,72.68,192.63
2026-06-30,1847.3910,430.0702,NaN,2.0018,NaN,543.3397,149.4188,62.3023,74.8665,17.9241,...,37.28,51.03,485.0431,690.11,2322.33,31.00,210.00,59.1310,74.15,193.61


In [5]:
INIT_DATE = pd.Timestamp("2015-05-13")
LAST_DATE = pd.Timestamp("2024-08-30")

TRAIN_START = INIT_DATE
TRAIN_END = pd.Timestamp("2017-12-31")

VAL_START = pd.Timestamp("2018-01-01")
VAL_END = pd.Timestamp("2018-12-31")

TEST_START = pd.Timestamp("2019-01-01")
TEST_END = LAST_DATE

In [6]:
daily_prices = daily_prices.loc[:, ASSETS].copy()
daily_prices.index = pd.to_datetime(daily_prices.index)
daily_prices.index.name = "date"
daily_prices.columns.name = "asset"

daily_prices = daily_prices.sort_index().loc[INIT_DATE:LAST_DATE]
daily_returns = daily_prices.pct_change(fill_method=None)

In [7]:
train_returns = daily_returns.loc[TRAIN_START:TRAIN_END]
valid_returns = daily_returns.loc[VAL_START:VAL_END]
test_returns = daily_returns.loc[TEST_START:TEST_END]

for name, df in [
    ("Train", train_returns),
    ("Valid", valid_returns),
    ("Test", test_returns)
]:
    print(
        f"{name:12s} | "
        f"{df.index.min().date()} - {df.index.max().date()} | "
        f"{df.shape}"
    )

Train        | 2015-05-13 - 2017-12-29 | (647, 30)
Valid        | 2018-01-02 - 2018-12-28 | (243, 30)
Test         | 2019-01-02 - 2024-08-30 | (1376, 30)


In [8]:
class BaselineModel:
    def __init__(self, num_assets: int, window_size: int = 1, window_type: str = "rolling"):
        assert window_size > 0
        assert window_type in ("rolling", "expanding")

        self.num_assets = num_assets
        self.window_size = window_size
        self.window_type = window_type

    def forward(self, x: np.ndarray) -> np.ndarray:
        # x: (days, num_assets)
        # out: (days - window_size + 1, num_assets)
        assert x.ndim == 2
        assert x.shape[1] == self.num_assets
        assert x.shape[0] >= self.window_size

        out = []
        for i in range(x.shape[0] - self.window_size + 1):
            end = i + self.window_size
            inputs = x[i:end] if self.window_type == "rolling" else x[:end]
            out.append(np.mean(inputs, axis=0))

        return np.stack(out, axis=0)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

# Training and Evaluation

In [9]:
# No training is required for the historical mean baseline

In [10]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    assert y_true.shape == y_pred.shape
    assert y_true.ndim == 2
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    errors = y_true - y_pred

    mse = np.mean(np.power(errors, 2))
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

    true_df = pd.DataFrame(y_true)
    pred_df = pd.DataFrame(y_pred)
    ic = pred_df.corrwith(true_df, axis=1, method="pearson")
    rank_ic = pred_df.corrwith(true_df, axis=1, method="spearman")

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Directional Accuracy": directional_accuracy,
        "IC": ic.mean(),
        "Rank IC": rank_ic.mean(),
        "ICIR": ic.mean() / ic.std(),
        "Rank ICIR": rank_ic.mean() / rank_ic.std()
    }

# Hyperparameter tuning

In [11]:
best_ic = -float("inf")
best_config = None

for window_type in ("rolling", "expanding"):
    for window_size in (1, 2, 4, 8, 16, 32, 64, 128, 7, 14, 21, 28):
        model = BaselineModel(num_assets=NUM_ASSETS, window_size=window_size, window_type=window_type)

        history = pd.concat([train_returns.tail(window_size), valid_returns])
        X_valid = history.iloc[:-1].to_numpy()
        y_valid_true = valid_returns.to_numpy()
        y_valid_pred = model(X_valid)

        valid_metrics = evaluate(y_valid_true, y_valid_pred)

        if valid_metrics["IC"] > best_ic:
            best_ic = valid_metrics["IC"]

            best_config = {
                "window_type": window_type,
                "window_size": window_size,
                "best_metric": best_ic.item(),
            }

display(best_config)

{'window_type': 'rolling',
 'window_size': 128,
 'best_metric': 0.010089216584001044}

# Out-Of-Sample

In [12]:
model = BaselineModel(NUM_ASSETS, window_size=best_config["window_size"], window_type=best_config["window_type"])

history = pd.concat([train_returns.tail(model.window_size), valid_returns, test_returns])
X_test = history.iloc[:-1].to_numpy()
y_test_true = test_returns.to_numpy()
y_test_pred = model(X_test)[-len(test_returns):]
test_metrics = evaluate(y_test_true, y_test_pred)
test_metrics

{'MSE': np.float64(0.0006661421482608435),
 'RMSE': np.float64(0.025809729720801873),
 'MAE': np.float64(0.01792743967732523),
 'Directional Accuracy': np.float64(0.47170542635658913),
 'IC': np.float64(-0.00468119917248741),
 'Rank IC': np.float64(-0.021939716231528197),
 'ICIR': np.float64(-0.015367575052179477),
 'Rank ICIR': np.float64(-0.08473920071080743)}

## Saving

In [13]:
test_predictions = pd.DataFrame(y_test_pred, index=test_returns.index, columns=test_returns.columns)
test_predictions.to_csv("baseline_oos_predictions.csv")
display(test_predictions)

asset,600169.SH,000049.SZ,600428.SH,002064.SZ,000501.SZ,603000.SH,000712.SZ,600580.SH,600017.SH,603355.SH,...,600718.SH,600507.SH,002583.SZ,601718.SH,600835.SH,002268.SZ,002092.SZ,600098.SH,002340.SZ,300001.SZ
date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,0.000099,0.000759,-0.000247,0.000017,-0.001719,-0.000569,0.000185,-0.000980,-0.000898,-0.001414,...,-0.000153,-0.000791,0.000627,-0.000660,-0.001363,0.000099,-0.002446,-0.000704,-0.003130,0.003943
2019-01-03,0.000065,0.000747,-0.000269,0.000054,-0.002040,-0.000537,0.000116,-0.001073,-0.001068,-0.001243,...,-0.000385,-0.000768,0.000497,-0.000778,-0.001472,-0.000097,-0.002393,-0.000684,-0.003296,0.003758
2019-01-04,-0.000003,0.000808,-0.000130,0.000531,-0.001792,-0.000291,0.000284,-0.001081,-0.001006,-0.001198,...,-0.000688,-0.000197,0.000307,-0.000655,-0.001531,-0.000203,-0.002255,-0.000826,-0.003153,0.003553
2019-01-07,0.000204,0.000924,0.000061,0.000642,-0.001560,0.000097,0.000989,-0.000788,-0.000838,-0.001230,...,-0.000450,0.000082,0.000745,-0.000430,-0.001207,-0.000098,-0.001810,-0.000697,-0.002838,0.003848
2019-01-08,0.000271,0.000653,0.000020,0.000580,-0.001571,-0.000055,0.001005,-0.000744,-0.000818,-0.001498,...,-0.000722,0.000209,0.000333,-0.000282,-0.001238,-0.000336,-0.002006,-0.000818,-0.002973,0.003780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-26,-0.001537,0.000966,0.000593,0.000763,-0.000644,-0.001859,0.000281,0.000684,0.000363,0.000599,...,0.000078,-0.001117,-0.000701,-0.001069,-0.000065,-0.003320,-0.002761,0.001064,0.001220,-0.000356
2024-08-27,-0.001338,0.001299,0.000740,0.000776,-0.000552,-0.002622,-0.000116,0.000902,0.000306,0.000480,...,0.000089,-0.001011,-0.000762,-0.000975,-0.000051,-0.003311,-0.002648,0.001052,0.001522,-0.000215
2024-08-28,-0.001423,0.000745,0.000651,0.000899,-0.000623,-0.002413,0.000009,0.000562,0.000306,0.000366,...,0.000016,-0.001014,-0.001054,-0.001064,-0.000181,-0.003590,-0.002744,0.001024,0.001387,-0.000593
